# GRIB 다중 지점 강수 시계열 추출

기본은 **`read_kma_grib`**(pygrib + 기상청 Lambert 격자)입니다. 파일명이 `DFS_*_PCP.*` 형태인 동네예보 강수에 맞습니다.

1. 지정 폴더에서 GRIB 파일 목록을 정리합니다.
2. 지점은 **`POINTS_CSV`**(예: `points_input.csv`)에서 읽고, 시계열 결과는 **`multipoint_rainfall_timeseries.csv`**에 지정한 6개 열만 저장합니다.

**패키지:** `pygrib`, `pandas`, `numpy` (스크립트 주석 참고).  
**`read_grib_xarray`(cfgrib)** 를 쓰려면 같은 환경에 `pip install eccodes cfgrib` 후 커널 재시작이 필요합니다.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from read_grib_timeseries import read_grib_xarray, read_kma_grib

## 1) 설정: 폴더, 파일 패턴, 지점 CSV(`POINTS_CSV`)

`points_input.csv` 형식: `point_name`(또는 `name`), `lat`, `lon`. 결과 `OUTPUT_CSV`에는 `point_name`, `base_time`, `forecast_hour`, `valid_time`, `variable`, `value`만 저장합니다 (`READ_METHOD="kma"` 기준).

In [2]:
# GRIB이 있는 폴더 (노트북 기준 상대 경로 또는 절대 경로)
DATA_FOLDER = Path("forecast_rainfall1")

# 예: 강수(PCP)만 — 필요에 따라 "*.grb*", "*TMP*" 등으로 변경
GLOB_PATTERN = "*PCP*"

# 지점 목록 CSV (UTF-8 BOM 가능). 열 이름 예:
#   point_name 또는 name, lat(또는 latitude), lon(또는 longitude)
POINTS_CSV = Path("points_input.csv")


def load_points_csv(path: Path | str) -> list[dict]:
    path = Path(path)
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]
    lower = {c.lower(): c for c in df.columns}

    def col(*candidates: str) -> str | None:
        for k in candidates:
            if k in lower:
                return lower[k]
        return None

    c_name = col("point_name", "name", "station", "site")
    c_lat = col("lat", "latitude", "y")
    c_lon = col("lon", "longitude", "lng", "x")
    if c_lat is None or c_lon is None:
        raise ValueError(f"{path}에 위도/경도 열이 필요합니다 (lat/lon 등). 현재 열: {list(df.columns)}")

    points: list[dict] = []
    for _, row in df.iterrows():
        lat = float(row[c_lat])
        lon = float(row[c_lon])
        if c_name is not None and pd.notna(row[c_name]) and str(row[c_name]).strip():
            name = str(row[c_name]).strip()
        else:
            name = f"{lat:.4f}_{lon:.4f}"
        points.append({"name": name, "lat": lat, "lon": lon})
    return points


POINTS = load_points_csv(POINTS_CSV)

# 결과 CSV 경로 (저장 열은 아래 OUTPUT_COLS 고정)
OUTPUT_CSV = Path("multipoint_rainfall_timeseries.csv")
OUTPUT_COLS = [
    "point_name",
    "base_time",
    "forecast_hour",
    "valid_time",
    "variable",
    "value",
]

# "kma": read_kma_grib (DFS/Lambert, pygrib) — cfgrib 없이 동작
# "xarray": read_grib_xarray — eccodes+cfgrib 설치 필요 (이 경우 아래 6열 저장과 맞지 않을 수 있음)
READ_METHOD = "kma"

## 2) 폴더 내 GRIB 파일 목록

In [3]:
def list_grib_files(folder: Path | str, pattern: str = "*") -> list[str]:
    """폴더에서 패턴에 맞는 파일 경로를 정렬된 리스트로 반환합니다."""
    folder = Path(folder)
    if not folder.is_dir():
        raise NotADirectoryError(folder)
    paths = sorted(p for p in folder.glob(pattern) if p.is_file())
    return [str(p) for p in paths]


file_list = list_grib_files(DATA_FOLDER, GLOB_PATTERN)
print(f"파일 개수: {len(file_list)}")
for i, fp in enumerate(file_list, 1):
    print(f"  {i:3d}. {Path(fp).name}")

파일 개수: 2
    1. DFS_SHRT_GRD_GRB4_PCP.202406212000
    2. DFS_SHRT_GRD_GRB5_PCP.202604072000


## 3) 지점별 시계열 추출 후 CSV 저장

`READ_METHOD`에 따라 `read_kma_grib` 또는 `read_grib_xarray`를 호출합니다. 저장 시 **`OUTPUT_COLS`에 있는 열만** `OUTPUT_CSV`에 기록합니다.

**주의:** 설정 셀(폴더·`POINTS`·`READ_METHOD`)을 먼저 실행하세요. 추출 셀만 단독 실행 시 `READ_METHOD`는 `"kma"`로 둡니다.

In [4]:
if not file_list:
    raise FileNotFoundError(f"'{DATA_FOLDER}' 에서 패턴 '{GLOB_PATTERN}' 에 맞는 파일이 없습니다.")

# 설정 셀을 건너뛴 경우 기본값 (DFS: kma)
READ_METHOD = globals().get("READ_METHOD", "kma")
OUTPUT_COLS = globals().get(
    "OUTPUT_COLS",
    [
        "point_name",
        "base_time",
        "forecast_hour",
        "valid_time",
        "variable",
        "value",
    ],
)

_m = str(READ_METHOD).lower()
if _m in ("kma", "dfs", "3", "pygrib_kma"):
    read_one = read_kma_grib
elif _m in ("xarray", "cfgrib", "1", "xr"):
    read_one = read_grib_xarray
else:
    raise ValueError(f"알 수 없는 READ_METHOD: {READ_METHOD!r} (예: 'kma' 또는 'xarray')")

POINTS = globals().get("POINTS")
if not POINTS:
    raise RuntimeError("POINTS가 없습니다. 설정 셀에서 POINTS_CSV를 읽어 주세요.")

frames: list[pd.DataFrame] = []
for pt in POINTS:
    name = pt.get("name") or f"{pt['lat']:.4f}_{pt['lon']:.4f}"
    df_pt = read_one(file_list, float(pt["lat"]), float(pt["lon"]))
    df_pt.insert(0, "point_name", name)
    frames.append(df_pt)

df_all = pd.concat(frames, ignore_index=True)
missing = [c for c in OUTPUT_COLS if c not in df_all.columns]
if missing:
    raise ValueError(
        f"저장할 열이 결과에 없습니다: {missing}. READ_METHOD='kma'(read_kma_grib)일 때 위 6열이 생성됩니다."
    )

df_out = df_all[OUTPUT_COLS].copy()
OUTPUT_CSV = globals().get("OUTPUT_CSV", Path("multipoint_rainfall_timeseries.csv"))
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_CSV.resolve()} (열: {list(df_out.columns)})")
df_out.head(10)

📍 요청 위치: (37.5665°N, 126.9780°E)
📍 격자 좌표: (x=60, y=127)
📍 실제 위치: (37.5799°N, 126.9894°E)

✅ 132개 시계열 데이터 추출 완료
📍 요청 위치: (35.1796°N, 129.0756°E)
📍 격자 좌표: (x=98, y=76)
📍 실제 위치: (35.1910°N, 129.0851°E)

✅ 132개 시계열 데이터 추출 완료
저장 완료: /home/hydro/research/dt/grib_reader/multipoint_rainfall_timeseries.csv (열: ['point_name', 'base_time', 'forecast_hour', 'valid_time', 'variable', 'value'])


,point_name,base_time,forecast_hour,valid_time,variable,value
0,서울시청,2024-06-21 20:00:00,6,2024-06-22 02:00:00,PCP,0.0
1,서울시청,2024-06-21 20:00:00,7,2024-06-22 03:00:00,PCP,0.0
2,서울시청,2024-06-21 20:00:00,8,2024-06-22 04:00:00,PCP,0.0
3,서울시청,2024-06-21 20:00:00,9,2024-06-22 05:00:00,PCP,0.0
4,서울시청,2024-06-21 20:00:00,10,2024-06-22 06:00:00,PCP,0.0
5,서울시청,2024-06-21 20:00:00,11,2024-06-22 07:00:00,PCP,0.0
6,서울시청,2024-06-21 20:00:00,12,2024-06-22 08:00:00,PCP,0.0
7,서울시청,2024-06-21 20:00:00,13,2024-06-22 09:00:00,PCP,0.0
8,서울시청,2024-06-21 20:00:00,14,2024-06-22 10:00:00,PCP,0.0
9,서울시청,2024-06-21 20:00:00,15,2024-06-22 11:00:00,PCP,0.0
